# 02 — Classical models (TF-IDF + stylometric features)

Trains (or loads) the five classical models behind the serving stack and evaluates them:
- **Linear SVM** and **Logistic Regression** on TF-IDF only (the style block measurably hurts linear solvers).
- **Naive Bayes**, **Random Forest** and **Gradient Boosting** on TF-IDF + the 10 stylometric features
  (trees use a chi2-selected 30k TF-IDF subset; the style columns always pass through).

Artifacts loaded by default: `data/tfidf_vectorizer_X.joblib`, `data/split_indices.joblib`, `models/*.joblib`.


In [ ]:
import json

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

from aitextd import paths
from aitextd.features import STYLE_FEATURE_NAMES, compute_style_features, style_features_vector

DATA = paths.PROJECT_ROOT / "data"
RANDOM_SEED = 42
TEST_SIZE = 0.2
TREE_FEATURE_COUNT = 30_000
FORCE_RETRAIN = False


## Canonical split and joint feature matrix

The matrix stacks TF-IDF columns first, then the 10 style features in the pinned
`STYLE_FEATURE_NAMES` order (the deployed sklearn checkpoints slice by that exact layout).


In [ ]:
X = joblib.load(DATA / "tfidf_vectorizer_X.joblib")
split = joblib.load(DATA / "split_indices.joblib")
train_idx, test_idx = split["train_idx"], split["test_idx"]

y = pd.read_csv(paths.PROCESSED_CSV, usecols=["aiGenerated"])["aiGenerated"].to_numpy()
y_train, y_test = y[train_idx], y[test_idx]
X_train, X_test = X[train_idx], X[test_idx]

vectorizer = joblib.load(paths.TFIDF_VECTORIZER_JOBLIB)
tfidf_dim = len(vectorizer.get_feature_names_out())
n_style = len(STYLE_FEATURE_NAMES)
assert X.shape[1] == tfidf_dim + n_style, (X.shape, tfidf_dim, n_style)
print(f"matrix {X.shape[0]} x {X.shape[1]} (tfidf {tfidf_dim} + style {n_style})")
print(f"train {len(train_idx)} / test {len(test_idx)} (seed {RANDOM_SEED}, test_size {TEST_SIZE})")


## Evaluation helper


In [ ]:
def expected_calibration_error(proba, y_true, n_bins=10):
    """ECE over 10 equal-width probability bins."""
    bins = np.minimum((proba * n_bins).astype(int), n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        mask = bins == b
        if mask.any():
            ece += mask.sum() / len(proba) * abs(y_true[mask].mean() - proba[mask].mean())
    return float(ece)


def evaluate(model, X_eval, y_eval):
    """Accuracy, per-class P/R/F1, ROC-AUC and (where probabilities exist) ECE."""
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_eval)[:, 1]
        y_pred = (score >= 0.5).astype(int)
        ece = expected_calibration_error(score, y_eval)
    else:  # LinearSVC exposes a raw margin, not probabilities
        score = model.decision_function(X_eval)
        y_pred = (score >= 0).astype(int)
        ece = None
    acc = accuracy_score(y_eval, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_eval, y_pred, average=None, labels=[0, 1], zero_division=0)
    return {
        "accuracy": round(float(acc), 4),
        "precision_human": round(float(p[0]), 4),
        "precision_ai": round(float(p[1]), 4),
        "recall_human": round(float(r[0]), 4),
        "recall_ai": round(float(r[1]), 4),
        "f1Score": round(float(f1[1]), 4),
        "rocAuc": round(float(roc_auc_score(y_eval, score)), 4),
        "ece": None if ece is None else round(ece, 4),
    }


## The five models

Load the deployed checkpoints, or retrain them with the exact hyperparameters of the
original pipeline when `FORCE_RETRAIN` is set.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

FILENAMES = {
    "svm": "svModel.joblib",
    "logreg": "logregModel.joblib",
    "naive_bayes": "nbModel.joblib",
    "random_forest": "rfModel.joblib",
    "gradient_boosting": "gbModel.joblib",
}


def tree_pipeline(clf):
    """chi2-select 30k TF-IDF columns; style columns always pass through."""
    return Pipeline(
        [
            (
                "select",
                ColumnTransformer(
                    [
                        ("tfidf_select", SelectKBest(chi2, k=TREE_FEATURE_COUNT), slice(0, tfidf_dim)),
                        ("style", "passthrough", slice(tfidf_dim, None)),
                    ],
                    sparse_threshold=1.0,
                ),
            ),
            ("clf", clf),
        ]
    )


def train_all():
    svm = LinearSVC(C=1.0, max_iter=5000, random_state=RANDOM_SEED)
    svm.fit(X_train[:, :tfidf_dim], y_train)
    logreg = LogisticRegression(solver="liblinear", dual=True, C=32.0, max_iter=5000, random_state=RANDOM_SEED)
    logreg.fit(X_train[:, :tfidf_dim], y_train)
    nb = MultinomialNB(alpha=0.01)
    nb.fit(X_train, y_train)
    rf = tree_pipeline(
        RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, random_state=RANDOM_SEED, class_weight="balanced")
    )
    rf.fit(X_train, y_train)
    gb = tree_pipeline(
        GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, random_state=RANDOM_SEED)
    )
    gb.fit(X_train, y_train)
    return {"svm": svm, "logreg": logreg, "naive_bayes": nb, "random_forest": rf, "gradient_boosting": gb}


if FORCE_RETRAIN:
    models = train_all()
else:
    models = {mid: joblib.load(paths.model_path(fname)) for mid, fname in FILENAMES.items()}
print("models ready:", ", ".join(models))


## Results on the canonical test split


In [ ]:
rows = {}
for mid, model in models.items():
    X_eval = X_test[:, :tfidf_dim] if mid in ("svm", "logreg") else X_test
    rows[mid] = evaluate(model, X_eval, y_test)
print(pd.DataFrame(rows).T.to_string())


## Sanity check on the v2 corpus

The deployed models were trained on the v1 dataset. `dataset_v2` is a later, larger,
leak-free rebuild — scoring the same models on its group-aware in-domain test split
shows how much the numbers move when the corpus construction changes.


In [ ]:
df_v2 = pd.read_parquet(DATA / "dataset_v2.parquet")
splits_v2 = joblib.load(DATA / "splits_v2.joblib")
style_v2 = pd.read_parquet(DATA / "style_features_v2.parquet")
assert list(style_v2.columns[1 : n_style + 1]) == STYLE_FEATURE_NAMES


def frame_features(frame):
    """Joint TF-IDF + style matrix for a v2 frame (style columns joined on doc_id)."""
    joined = frame[["doc_id"]].merge(style_v2, on="doc_id", how="left")
    return sparse.hstack([vectorizer.transform(frame["text"]), joined[STYLE_FEATURE_NAMES].to_numpy()], format="csr")


v2_test = df_v2.iloc[splits_v2["in_domain_group__test"]]
X_v2, y_v2 = frame_features(v2_test), v2_test["label_binary"].to_numpy()
print(f"v2 in-domain test rows: {len(v2_test)}")

rows_v2 = {}
for mid, model in models.items():
    rows_v2[mid] = evaluate(model, X_v2[:, :tfidf_dim] if mid in ("svm", "logreg") else X_v2, y_v2)
print(pd.DataFrame(rows_v2).T.to_string())


## Paraphrase attack

`data/paraphrase_attack.parquet` holds 1,200 AI texts after humanizing paraphrases.
The metric below is the fraction still detected as AI — how much detection survives.


In [ ]:
attack = pd.read_parquet(DATA / "paraphrase_attack.parquet")
attack_style = np.vstack([style_features_vector(compute_style_features(t, t, vectorizer)) for t in attack["text"]])
X_att = sparse.hstack([vectorizer.transform(attack["text"]), attack_style], format="csr")
y_att = attack["label_binary"].to_numpy()

survival = {}
for mid, model in models.items():
    res = evaluate(model, X_att[:, :tfidf_dim] if mid in ("svm", "logreg") else X_att, y_att)
    survival[mid] = res["recall_ai"]
print(pd.Series(survival, name="ai_recall_under_attack").round(4).to_string())


## Registration (only with FORCE_RETRAIN)

Dumps each retrained model into `models/` and rewrites its `models.json` entry
in the serving format; hand-tuned ensemble weights survive a retrain.


In [ ]:
def commit(model, entry):
    joblib.dump(model, paths.model_path(entry["filename"]))
    with open(paths.MODELS_JSON, encoding="utf-8") as f:
        existing = json.load(f)
    prior = next((m for m in existing if m["id"] == entry["id"]), None)
    entry.setdefault("ensembleWeight", (prior or {}).get("ensembleWeight", 1.0))
    existing = [m for m in existing if m["id"] != entry["id"]] + [entry]
    with open(paths.MODELS_JSON, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)
    print(f"saved {entry['filename']} + models.json entry '{entry['id']}'")


def entry_fields(result, feature_count, n_style_cols):
    return {
        "type": "ml",
        "version": "4.0",
        "runtime": "sklearn-tfidf",
        "vectorizer": "vectorizer.joblib",
        "styleFeatureCount": n_style_cols,
        **result,
        "training": {
            "totalSamples": len(train_idx) + len(test_idx),
            "trainSamples": len(train_idx),
            "testSamples": len(test_idx),
            "featureCount": feature_count,
            "testSplitRatio": TEST_SIZE,
            "randomSeed": RANDOM_SEED,
            "dataset": "persian_ai_generated_final.csv + blogs.csv",
        },
    }


if FORCE_RETRAIN:
    for mid, model in models.items():
        n_cols = 0 if mid in ("svm", "logreg") else n_style
        feature_count = tfidf_dim if mid in ("svm", "logreg") else (
            TREE_FEATURE_COUNT + n_style if mid in ("random_forest", "gradient_boosting") else tfidf_dim + n_style
        )
        entry = {
            "id": mid,
            "filename": FILENAMES[mid],
            "metrics": {k: v for k, v in rows[mid].items() if v is not None},
            **entry_fields(rows[mid], feature_count, n_cols),
        }
        commit(model, entry)
else:
    print("FORCE_RETRAIN is False — deployed models.json untouched")


## Summary


In [ ]:
records = []
for split_name, table in (("v1_test", rows), ("v2_in_domain", rows_v2)):
    for mid, res in table.items():
        records.append({"model": mid, "split": split_name, **res})
print(pd.DataFrame(records).set_index(["model", "split"]).to_string())
